# Universe Filter Playground
Tier 1 weekly filter: large-cap US stocks → `data/watchlist.csv`

In [1]:
# Cell 1 — Setup & import
import sys
sys.path.insert(0, 'backend/01_scanner')

from universe_filter import (
    get_max_share_price,
    get_earnings_tickers,
    run_universe_filter,
    save_watchlist,
)
import pandas as pd

print('Imports OK')

Imports OK


In [2]:
# Cell 2 — Happy path: run full filter and inspect output
count = run_universe_filter()
print(f'\nResult: {count} stocks saved')

if count:
    df = pd.read_csv('data/watchlist.csv')
    display(df.head(10))
    print(f'\nColumns: {df.columns.tolist()}')
    print(f'Price range: ${df["price"].min():.2f} – ${df["price"].max():.2f}')
    print(f'ATR% range:  {df["atr_pct"].min():.1f}% – {df["atr_pct"].max():.1f}%')

[universe] Alpaca unavailable, using fallback ceiling $360.00: ('Key ID must be given to access Alpaca trade API', ' (env: APCA_API_KEY_ID)')
[universe] price ceiling: $360.00
[universe] querying TradingView screener...
[universe] 70 stocks after ATR% filter (1.0%–5.0%)
[universe] 5 stocks removed for upcoming earnings
[universe] saved 65 tickers to data/watchlist.csv

Result: 65 stocks saved


,ticker,price,volume,atr,atr_pct
0,PLTR,133.99,51277898,6.11,4.56
1,AAPL,300.23,48309889,6.24,2.08
2,AMZN,264.14,39289953,6.66,2.52
3,PFE,25.33,39084328,0.55,2.17
4,NFLX,87.02,36334330,2.58,2.97
5,BAC,49.77,34416403,1.15,2.32
6,T,24.03,34369037,0.61,2.53
7,CSCO,118.21,31023440,3.58,3.03
8,ORCL,192.95,23073934,9.09,4.71
9,UBER,75.09,20429384,2.55,3.40



Columns: ['ticker', 'price', 'volume', 'atr', 'atr_pct']
Price range: $24.03 – $326.31
ATR% range:  1.6% – 4.9%


In [3]:
# Cell 3 — Parameter variations

# Vary earnings window: how many stocks get removed at different lookforward windows?
for days in [3, 5, 10]:
    tickers = get_earnings_tickers(days_ahead=days)
    count = len(tickers) if tickers else 0
    print(f'days_ahead={days}: {count} earnings events')

# Show current price ceiling
print(f'\nCurrent price ceiling: ${get_max_share_price():.2f}')

days_ahead=3: 248 earnings events
days_ahead=5: 295 earnings events
days_ahead=10: 405 earnings events
[universe] Alpaca unavailable, using fallback ceiling $360.00: ('Key ID must be given to access Alpaca trade API', ' (env: APCA_API_KEY_ID)')

Current price ceiling: $360.00


In [4]:
# Cell 4 — Failure path: confirm graceful degradation
import os

# Temporarily remove Finnhub key — should skip earnings filter, not crash
original_key = os.environ.pop('FINNHUB_API_KEY', None)
result = get_earnings_tickers()
print(f'Missing Finnhub key → returned: {result}  (expected: None)')

# Restore key
if original_key:
    os.environ['FINNHUB_API_KEY'] = original_key

# Bad URL in earnings (simulate network failure)
import unittest.mock as mock
with mock.patch('requests.get', side_effect=Exception('network error')):
    result = get_earnings_tickers()
    print(f'Network failure → returned: {result}  (expected: None)')

[universe] FINNHUB_API_KEY not set, skipping earnings filter
Missing Finnhub key → returned: None  (expected: None)
[universe] Finnhub unavailable, skipping earnings filter: network error
Network failure → returned: None  (expected: None)


In [ ]:
# Cell 5 — Free play
